## Chapter2 : Working with Text data

In [1]:
from importlib.metadata import version
# The importlib module in Python provides the implementation of the import statement and the __import__() function, offering a programmatic way to interact with Python's import system. It exposes components that allow for dynamic module loading, customization of the import process, and access to package metadata.

print("Torch version: ", version("torch"))
print("Torchvision version: ", version("torchvision"))


Torch version:  2.9.0+cu126
Torchvision version:  0.24.0+cu126


This chapter covers the data preperation and sampling to get input data "ready" for LLM.

### Understanding Word Embedding
**There are many forms of embeddings, such as * Video Embeddings, * Audio Embeddings and * text embeddings , we focus on the text embeddings here...

In [2]:
# Tokenizing the text : Breaking the text into smaller units
# such as individual words, and punctuation characters

import os
import requests

if not os.path.exists("the-verdict.txt"):
  url = (
      "https://raw.githubusercontent.com/rasbt/"
      "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
      "the-verdict.txt"
)

  file_path = "the-verdict.txt"

  response = requests.get(url, timeout=30)
  response.raise_for_status()
  with open(file_path, "wb") as f:
    f.write(response.content)






In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
import re
# THis will split the text into whitespaces


text = "Hello, world, This is a test"
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world,', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test']


Here we split the text on whitespaces but we also want commas and periods , so we modify the regex: to do that as well.....

In [5]:
result = re.split(r'[,.]||\s', text)

print(result)

['', 'H', 'e', 'l', 'l', 'o', '', '', '', 'w', 'o', 'r', 'l', 'd', '', '', '', 'T', 'h', 'i', 's', '', '', 'i', 's', '', '', 'a', '', '', 't', 'e', 's', 't', '']


And as we can see it make empty strings, so we remove them ....

In [6]:
# strip whitespace from each item and then filter out any empty string,,,,,

result = [item for item in result if item.strip()]
print(result)

['H', 'e', 'l', 'l', 'o', 'w', 'o', 'r', 'l', 'd', 'T', 'h', 'i', 's', 'i', 's', 'a', 't', 'e', 's', 't']



* The above is good but we need to add other types of punctuations, such as periods, questions marks and so on..

In [7]:
text = "Hello , world, Is this -- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

['Hello', ',', 'world', ',', 'Is', 'this', '--', 'a', 'test', '?']


In [8]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


* Calculating the total number of tokens


In [9]:
print(len(preprocessed))

4690


## Converting token into token ids...

* Next, we convert the text tokens into token IDs that we can process via embedding layers later
* From these tokens, we can now build a vocabulary that consists of all the uniwue tokens



In [10]:
all_words = sorted(set(preprocessed))

vocab_size = len(all_words)

print(vocab_size)

1130


In [11]:
vocab = {token:integer for integer , token in enumerate(all_words)}

* The last 50 entries in this vocabulary


In [12]:
# for i, item in enumerate(list(vocab.items())[:50]):
#     print(item)
for i, item in enumerate(list(vocab.items())):
    print(item)

    if i >= 50:
      break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


* Here we put all these into a tokenizer class ...

In [13]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


The `encode` function tunrs the text into token IDs
and `decode` function tunrs the token back into text

* We can use the tokenizer to encode (that is, tokenize) texts into integers
* These integers can then be embedded (later) as input of/for the LLM

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

text = """ "It's the last he painted, you know."
  Mrs. Gisburn said with pardonable pride. """

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 7, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


* We can decode the integers back into text


In [15]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know." Mrs. Gisburn said with pardonable pride.'

In [16]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know." Mrs. Gisburn said with pardonable pride.'

### Adding special  context tokens ...

* It's useful to add some "special" tokens for unknown words and to denote the end of a text


* Some tokenizers use special tokens to help the LLM  with additional context..


* SOme od these special tokens are:
`[BOS]` (beginning of the sequence) marks the beginning of text.
`[EOS]` (end of sequence) marks where the text ends ( this is usually used to cancatenate multiple unrealted texts, two different wikipedia articles or 2 different books and so on)

`[PAD]` (padding ) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length

`[UNK]` :to represent words that are not included in the vocabulary.


Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity


The `<|endoftext|>` is analogous to the [EOS] token mentioned above.

GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)

GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding `(BPE)` tokenizer, which breaks down words into subword units which we will discuss in a later section.



  We use the `<|endoftext|>`tokens between two independent sources of text:





In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)


KeyError: 'Hello'

* The above produces an error because the word "Hello" is not contained in the vocabulary

* To deal with such cases, we can add special tokens like "<|unk|>" to the vocabulary to represent unknown words

* Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)


In [18]:
all_tokens = sorted(list(set(preprocessed)))

all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer , token in enumerate (all_tokens)}

In [19]:
len(vocab.items())

1132

In [21]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


* We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token...

In [20]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text


Let tokenize text with modified tokenizer


In [22]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## BytePair Encoding ....

* GPT-2 used BytePair encoding (BPE) as its tokenizer

* it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words

* For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges

* In this chapter, we are using the BPE tokenizer from OpenAI's open-source tiktoken library, which implements its core algorithms in Rust to improve computational performance





In [23]:
 !pip install tiktoken


In [24]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [28]:
tokenizer = tiktoken.get_encoding("gpt2")

In [29]:
text = (
    "Hello do you like tea? <|endoftext|> In the Sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 3825, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [30]:
strings = tokenizer.decode(integers)

print(strings)

Hello do you like tea? <|endoftext|> In the Sunlit terracesof someunknownPlace.


## Data Sampling with Sliding window ....

* We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:



In [31]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))


5145


* For each text chunk, we want the inputs and targets

* Since we want to predict the next word, the targets are the inputs shifted by one position to the right ...

In [33]:
enc_sample = enc_text[50:]

In [34]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size]

print(f"x: {x}")
print(f"y:             {y}")

x: [290, 4920, 2241, 287]
y:             [4920, 2241, 287]


* one by one the prediction will look like as this ...

In [35]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]


  print(context, '----->', desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [36]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context), '------>', tokenizer.decode([desired]))

 and ------>  established
 and established ------>  himself
 and established himself ------>  in
 and established himself in ------>  a


* Next : we implement a simple data laoder that iterate over the input dataset and returns the inputs and target shifted by one.


In [37]:
import torch
print("PyTorch_version", torch.__version__)


PyTorch_version 2.9.0+cu126


* We use SlidingWindow approach , changing position by +1

* Create the dataset and dataloader that extract chunks from input text dataset is the next step .....

In [39]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # Tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
    assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

    # Use a sliding window to chunk the book into overlapping sequences of max_length

    for i in range(0, len(token_ids) - max_length , stride):
      input_chunk = token_ids[i:i+ max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))


  def   __len__(self):
    return len(self.input_ids)

  def  __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]






In [40]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
  # Initialize the tokenizer

  tokenizer = tiktoken.get_encoding("gpt2")

  # Crete a dataset
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  # Create the DataLoader
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers
  )


  return dataloader



 * Let test the dataloder with a batch size of 1 for an LLM with a context size of 4:

In [41]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

In [42]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [43]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


* As can be seen the window slides with a length of 1,

* Next we can increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting
.....


In [45]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4,
                                  stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs\n", inputs)
print("Targets\n", targets)

Inputs
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating token embeddings ....

* The data is already almost ready for an LLM
* BUt lastlt let us embed the tokens in a continuous vector representation using an embedddig layer
* Usually these embeddings layers are part of the LLM itself and are updated ( traind) during the model training


* Supoose we have the following four inputs examples with inputs ids 2, 3, 5 and 1 ( after toknization )


In [46]:
input_ids  = torch.tensor([2, 3, 5, 1])

* For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:
*

In [49]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

* This would result in a 6x3 weight matrix:


In [50]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


* For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected layer,

* Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation

* to convert a token with id 3 into a 3d vector we do the following...

In [51]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


* Note that the above is the 4th row in the `embedding_layer` weight matrix

* To embed all four `input_ids` values above, we do:

In [52]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


* AN embedding layer is essetially a lookup operation .............

##  Encoding word positions

* Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence:


* Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:

* The BytePair encoder has a vocabulary size of 50,257:

* Suppose that we want tot encode the input tokens into a 256-dim: vector representation ..


In [53]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

* If we sample from the dataloader we embed the tokens in each batch into a 256-dim: vector

* If we have a batch of size 8 with 4 tokens each this results in a 8 x 4 x 256 tensor ..


In [54]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)


In [55]:
print("Token IDs:\n ", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [57]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
         [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
         [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
         [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],

        [[ 1.5460,  1.7368, -0.7848,  ..., -0.1004,  0.8584, -0.3421],
         [-1.8622, -0.1914, -0.3812,  ...,  1.1220, -0.3496,  0.6091],
         [ 1.9847, -0.6483, -0.1415,  ..., -0.3841, -0.9355,  1.4478],
         [ 0.9647,  1.2974, -1.6207,  ...,  1.1463,  1.5797,  0.3969]],

        [[-0.7713,  0.6572,  0.1663,  ..., -0.8044,  0.0542,  0.7426],
         [ 0.8046,  0.5047,  1.2922,  ...,  1.4648,  0.4097,  0.3205],
         [ 0.0795, -1.7636,  0.5750,  ...,  2.1823,  1.8231, -0.3635],
         [ 0.4267, -0.0647,  0.5686,  ..., -0.5209,  1.3065,  0.8473]],

        ...,

        [[-1.6156,  0.9610, -2.6437,  ..., -0.9645,  1.0888,  1.6383],
         [-0.3985, -0.9235, -1.31

* GPT-2 uses absolute position embeddings, so we just create another embedding layer


In [58]:
context_length = max_length

pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

print(pos_embedding_layer)

Embedding(4, 256)


In [59]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

print(pos_embeddings)

torch.Size([4, 256])
tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       grad_fn=<EmbeddingBackward0>)


* To create input embeddings used in an LLM , we simply add the token and the positional embedddings



In [60]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

print(input_embeddings)


torch.Size([8, 4, 256])
tensor([[[ 2.2288,  0.5619,  0.8286,  ..., -0.6272, -0.2987,  0.8900],
         [ 2.0903, -0.4664, -0.0593,  ...,  0.9115, -1.0493, -1.6473],
         [-0.7158, -0.8304,  1.2494,  ...,  2.3952,  1.8773,  0.8051],
         [ 0.2703,  0.4029,  3.0514,  ...,  0.3595, -1.4548,  0.8310]],

        [[ 3.2835,  1.1749, -1.4150,  ..., -0.3281,  2.4332,  0.6924],
         [-0.2199, -0.9114, -0.1750,  ...,  1.5337, -0.1998,  0.1462],
         [ 1.5197, -1.4240,  0.4391,  ...,  1.0494, -1.4318,  2.3057],
         [ 0.2893,  0.8346, -0.1884,  ...,  1.9602,  0.8709,  0.8796]],

        [[ 0.9662,  0.0952, -0.4640,  ..., -1.0320,  1.6290,  1.7771],
         [ 2.4468, -0.2154,  1.4984,  ...,  1.8766,  0.5595, -0.1423],
         [-0.3856, -2.5393,  1.1556,  ...,  3.6157,  1.3267,  0.4944],
         [-0.2487, -0.5275,  2.0009,  ...,  0.2930,  0.5977,  1.3300]],

        ...,

        [[ 0.1219,  0.3991, -3.2740,  ..., -1.1921,  2.6637,  2.6728],
         [ 1.2438, -1.6436, -1.11

* In the initial phase of the input processing workflow, the input text is segmented into separate tokens

* Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:
